# WATCHDOG Main-Modul Dokumentation

Dieses Notebook dokumentiert die Datei `watchdog/main.py`, den zentralen Einstiegspunkt der WATCHDOG-Anwendung.

## Zweck

Das Main-Modul orchestriert die Hauptkomponenten des Systems:

- Logging initialisieren
- Datenbank verbinden
- Modbus-Kommunikation aufbauen
- Messwerte zyklisch auslesen
- Messwerte speichern
- Fehler behandeln
- Ressourcen sauber freigeben

In [ ]:
from watchdog.main import run

# Einstiegspunkt
# run()

## Ablaufdiagramm

```text
Start
 │
 ├─ setup_logger()
 │
 ├─ Datenbank verbinden
 │
 ├─ Modbus verbinden
 │
 ├─ Zyklische Schleife
 │    ├─ Register lesen
 │    ├─ Werte speichern
 │    ├─ Werte loggen
 │    └─ Sleep(Polling-Intervall)
 │
 └─ Aufräumen (finally)
      ├─ Modbus schließen
      └─ Datenbank schließen
```

## Abhängigkeiten

```python
from watchdog.config import MODBUS_CONFIG
from watchdog.database import WatchdogDatabase
from watchdog.logger import setup_logger
from watchdog.modbus_client import WatchdogModbusClient
```

### Verantwortlichkeiten

- `config.py` → zentrale Konfiguration
- `database.py` → Persistierung von Messwerten
- `logger.py` → Logging-Konfiguration
- `modbus_client.py` → Kommunikation mit Feldgeräten

## Funktion run()

Die Funktion `run()` bildet den kompletten Runtime-Lifecycle der Anwendung ab.

### Initialisierung

- Logger starten
- Konfiguration protokollieren
- Datenbankobjekt erzeugen
- Modbus-Client erzeugen

## Datenfluss

```text
Modbus Gerät
      │
      ▼
WatchdogModbusClient
      │
      ▼
read_all_registers()
      │
      ▼
Dictionary mit Messwerten
      │
      ▼
WatchdogDatabase.insert_measurements()
      │
      ▼
SQLite Datenbank
```

## Fehlerbehandlung

### Kommunikationsfehler

Fehler beim Lesen oder Speichern werden abgefangen:

```python
except Exception as error:
    logging.error(...)
```

Der Watchdog läuft anschließend weiter.

### Benutzerabbruch

```python
except KeyboardInterrupt
```

Ermöglicht kontrolliertes Beenden über STRG+C.

## Ressourcenmanagement

Der `finally`-Block sorgt dafür, dass Ressourcen immer freigegeben werden:

```python
client.close()
database.close()
```

Dadurch werden Datei-Handles und serielle Schnittstellen korrekt geschlossen.

## ADR-003: Polling-basierte Architektur

### Status
Accepted

### Entscheidung
WATCHDOG verwendet eine einfache Polling-Schleife für die Datenerfassung.

### Gründe
- Einfach implementierbar
- Gut nachvollziehbar
- Robust auf Embedded-Systemen
- Keine zusätzlichen Frameworks

### Vorteile
- Geringe Komplexität
- Einfache Fehlersuche
- Deterministisches Verhalten

### Nachteile
- Permanente Abfrage auch ohne Änderungen
- Begrenzte Skalierung bei vielen Geräten

### Zukunft
- Scheduler
- AsyncIO
- Multi-Device Polling
- Health Monitoring

## Verbesserungsvorschläge

1. Graceful Shutdown via Signal Handler
2. Batch-Speicherung von Messwerten
3. Heartbeat-Monitoring
4. Retry-Mechanismus für Modbus
5. Health-Check-Endpunkt
6. Metriken für Diagnose und Fernwartung